# 04 - Qualitative coding

Combines the LLM judge's `type` labels with the human annotators' codings
to produce the qualitative codebook in the paper:

* % of capitulating responses that **paraphrase** the attacker's reasoning
* % that **invent independent** justifications
* % that **simply concede** without reasoning

Reports both judge labels (large N) and human labels (50, with kappa).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import pandas as pd
from src.utils.io import load_jsonl

In [ ]:
RUNS_DIR = pathlib.Path('../runs')
judge = []
for jf in RUNS_DIR.rglob('judge.jsonl'):
    judge.extend(load_jsonl(jf))
j = pd.DataFrame(judge)
j['judge_type'].value_counts(normalize=True)

In [ ]:
# Type breakdown by attack condition (judge labels)
breakdown = (j[j['judge_capitulated']]
             .groupby(['condition', 'judge_type']).size().unstack(fill_value=0))
breakdown_pct = breakdown.div(breakdown.sum(axis=1), axis=0) * 100
breakdown_pct.round(1)

In [ ]:
# Same breakdown, human labels (when present)
human_csvs = list(RUNS_DIR.rglob('annotations_*.csv'))
if human_csvs:
    hs = pd.concat([pd.read_csv(c) for c in human_csvs], ignore_index=True)
    hs = hs[hs['capitulated'].astype(str) == '1']
    print('Human-coded breakdown by attack condition:')
    print((hs.groupby(['condition', 'type']).size().unstack(fill_value=0)
             .pipe(lambda d: d.div(d.sum(axis=1), axis=0) * 100).round(1)))
else:
    print('No human annotations yet -- generate them via scripts/make_human_eval.py')